In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# =========================
# LOAD DATA
# ========================

# change the path name.
df = pd.read_csv("/content/train.csv")
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# =========================
# ENCODE CATEGORICAL
# =========================
for col in X.columns:
    if X[col].dtype == "object":
        X[col] = LabelEncoder().fit_transform(X[col])

if y.dtype == "object":
    y = LabelEncoder().fit_transform(y)

# =========================
# IMPUTATION + SCALING
# =========================
X = SimpleImputer(strategy="mean").fit_transform(X)
X = StandardScaler().fit_transform(X)

# =========================
# EXPERIMENTS
# =========================
results = []

experiments = [
    (1.0, "100% Training Data"),
    (0.2, "20% (Mini) Training Data")
]

for ratio, label in experiments:
    if ratio == 1.0:
        # Use the full dataset
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
    else:
        # First downsample to 20% -> mini dataset
        X_small, _, y_small, _ = train_test_split(
            X, y, test_size=0.8, random_state=42, stratify=y
        )
        # Then split that mini dataset into train/val
        X_train, X_val, y_train, y_val = train_test_split(
            X_small, y_small, test_size=0.2, random_state=42, stratify=y_small
        )

    # =========================
    # SVM
    # =========================
    if ratio == 1.0:
        svm = SVC(
            kernel="rbf",
            C=0.01,
            gamma="scale",
            random_state=42
        )
    else:
        svm = SVC(
            kernel="rbf",
            C=100.0,
            gamma="scale",
            random_state=42
        )

    svm.fit(X_train, y_train)
    svm_pred = svm.predict(X_val)
    svm_f1 = f1_score(y_val, svm_pred, average="macro")

    # =========================
    # NEURAL NETWORK
    # =========================
    if ratio == 1.0:
        nn = MLPClassifier(
            hidden_layer_sizes=(512, 256, 128),
            activation="relu",
            solver="adam",
            alpha=1e-5,
            batch_size=256,
            learning_rate="adaptive",
            max_iter=4000,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=30,
            random_state=42
        )
    else:
        nn = MLPClassifier(
            hidden_layer_sizes=(8,),
            activation="tanh",
            solver="sgd",
            learning_rate_init=0.001,
            learning_rate="constant",
            alpha=10.0,
            batch_size=256,
            max_iter=10,
            tol=1e-2,
            early_stopping=False,
            random_state=42
        )

    nn.fit(X_train, y_train)
    nn_pred = nn.predict(X_val)
    nn_f1 = f1_score(y_val, nn_pred, average="macro")

    better = "NN" if nn_f1 > svm_f1 else "SVM"

    results.append([
        label,
        round(svm_f1, 4),
        round(nn_f1, 4),
        better
    ])

results_df = pd.DataFrame(
    results,
    columns=["Dataset Used", "SVM Macro F1", "NN Macro F1", "Better Model"]
)
print(results_df)


               Dataset Used  SVM Macro F1  NN Macro F1 Better Model
0        100% Training Data        0.7135       0.7252           NN
1  20% (Mini) Training Data        0.6469       0.6323          SVM


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(
